In [1]:

from pathlib import Path
import shutil

APP_DIR = Path("streamlit_app")
APP_DIR.mkdir(parents=True, exist_ok=True)
APP_PATH = APP_DIR / "app.py"
MODEL_PATH = Path("best_model.onnx")

if not MODEL_PATH.exists():
    raise FileNotFoundError("best_model.onnx not found. Export it from notebooks/dsai/phase2_mlflow_nn.ipynb.")

shutil.copy2(MODEL_PATH, APP_DIR / "best_model.onnx")

app_code = """import numpy as np
import pandas as pd
import streamlit as st
import onnxruntime as ort

MODEL_PATH = 'best_model.onnx'

# Update to match the NN input size from ml.ipynb
INPUT_SIZE = 200

st.set_page_config(page_title='NN Predictor', layout='wide')
st.title('NN Predictor')
st.caption('ONNX model loaded from best_model.onnx')

session = ort.InferenceSession(MODEL_PATH)

rows = (INPUT_SIZE + 3) // 4
columns = ['time', 'x', 'y', 'z']
if 'inputs_df' not in st.session_state:
    st.session_state['inputs_df'] = pd.DataFrame(0.0, index=range(rows), columns=columns)

st.subheader('Inputs')
edited_df = st.data_editor(
    st.session_state['inputs_df'],
    use_container_width=True,
    num_rows='fixed',
)

flat = edited_df[columns].to_numpy().reshape(-1)[:INPUT_SIZE]
input_array = np.array([flat], dtype=np.float32)
input_name = session.get_inputs()[0].name
output = session.run(None, {input_name: input_array})[0]

st.subheader('Prediction')
st.write(output.tolist()[0])
"""

APP_PATH.write_text(app_code, encoding="utf-8")

print(f"Wrote Streamlit app to {APP_PATH}")

Wrote Streamlit app to streamlit_app/app.py


In [ ]:
!cd streamlit_app && streamlit run app.py